# 🚀 Modul 08 Hands-on: Model Serving & Production API Microservice

Di hands-on puncak ini, Anda akan mempraktikkan proses deployment model ke server production:
1. **Export ke ONNX (Open Neural Network Exchange)**: Membebaskan model dari dependensi PyTorch interpreter.
2. **Latency Benchmarking**: Mengukur kecepatan inferensi model.
3. **FastAPI Microservice**: Membedah struktur file server produksi `app.py`.
4. **API Testing**: Menguji request inferensi JSON menggunakan `TestClient` dan melihat response prediksinya!

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import os

# Definisikan arsitektur model klasifikasi churn
class ChurnModel(nn.Module):
    def __init__(self, in_features=6, hidden_dim=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
    def forward(self, x):
        return self.network(x)

model = ChurnModel()
model.eval()
print("Model siap untuk proses export dan serving!")

## 1. Export PyTorch Model ke Format Standar Industri ONNX
Gunakan `torch.onnx.export` dengan dummy input untuk men-trace grafik komputasi.

In [ ]:
ONNX_FILE_PATH = "churn_model.onnx"
dummy_input = torch.randn(1, 6, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_FILE_PATH,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input_features"],
    output_names=["output_logits"],
    dynamic_axes={
        "input_features": {0: "batch_size"},
        "output_logits": {0: "batch_size"}
    }
)

file_size_kb = os.path.getsize(ONNX_FILE_PATH) / 1024
print(f"✅ Model berhasil di-export ke: {ONNX_FILE_PATH}")
print(f"Ukuran file ONNX: {file_size_kb:.2f} KB (Sangat ringan dan siap kirim ke production!)")

## 2. Benchmark Kecepatan Inferensi
Mari kita uji berapa milidetik yang dibutuhkan model untuk melayani 1.000 prediksi pelanggan berturut-turut.

In [ ]:
test_input = torch.randn(1, 6)
num_requests = 1000

# Warmup
for _ in range(50):
    _ = model(test_input)

start_time = time.time()
with torch.no_grad():
    for _ in range(num_requests):
        _ = model(test_input)
total_time = time.time() - start_time

avg_latency_ms = (total_time / num_requests) * 1000
throughput = num_requests / total_time
print(f"⏱️ Rata-rata Latensi Inferensi: {avg_latency_ms:.3f} ms per request")
print(f"⚡ Throughput: {throughput:.0f} requests per detik!")

## 3. Testing REST API Server (FastAPI TestClient)
Kita uji file `app.py` yang berada di folder yang sama menggunakan `TestClient` bawaan FastAPI.

In [ ]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

# 1. Test Endpoint Health Check
response_health = client.get("/health")
print("1. Response Health Check:", response_health.json())

# 2. Test Endpoint Predict dengan Payload Pelanggan
sample_payload = {
    "age": 42,
    "tenure": 5,
    "monthly_charges": 95.0,
    "total_spend": 475.0,
    "support_tickets": 3,
    "online_services": 2
}

response_predict = client.post("/predict", json=sample_payload)
print("\n2. Response Prediksi JSON:")
for k, v in response_predict.json().items():
    print(f"   {k:20s}: {v}")

print("\n💡 Sukses! API microservice siap dijalankan di terminal dengan perintah:")
print("   uvicorn app:app --reload --port 8000")
print("   Lalu buka Swagger UI di browser: http://127.0.0.1:8000/docs")